# Kapitel 3 – Övningsuppgifter: Regression

Det här är mina svar på övningsuppgifterna till kapitel 3 (Regression) i boken *"Lär dig AI från grunden - Tillämpad maskininlärning med Python"* (Prgomet, Johnson, Solberg, Rundberg Streuli). Frågorna kommer från övningsuppgifterna i bokens GitHub-repo.

## Fråga 1 – Vad kännetecknar regressionsproblem?

Enligt avsnitt 3.1 handlar regressionsproblem om att man vill predikera en **kontinuerlig beroende variabel** ($y$) utifrån ett antal oberoende variabler ($x$). Det som skiljer regression från klassificering är just det här – utdatan är kontinuerlig, inte diskret som i ett klassificeringsproblem.

Boken tar upp ett antal exempel på hur det kan se ut i praktiken:

- Bilvärdering: predikera försäljningspriset på en bil utifrån miltal, ålder, bränsletyp och märke.
- Fastighetsvärdering: predikera priset på en fastighet utifrån storlek, byggnadsår, läge och antal rum.
- Jordbruksproduktion: predikera hur stor skörden (t.ex. potatis i kilogram) blir utifrån temperatur, regnmängd och använd gödningsmedel.
- Försäkringar: predikera förväntad skadekostnad för en kund utifrån ålder, utbildning, läge och tidigare historik.
- Efterfrågan på produkter: predikera hur många enheter som kommer säljas utifrån pris, säsong, kampanjer och tidigare försäljningsdata.
- Lönennivå: predikera en persons lön utifrån utbildningsnivå, ålder, erfarenhet, bransch och geografiskt område.

Det som är gemensamt för alla exemplen är att målvariabeln $y$ är ett tal på en kontinuerlig skala – kronor, kilogram, antal enheter och så vidare – och inte en kategori.

## Fråga 2 – Förklara utvärderingsmåtten RMSE, MSE och MAE

Dessa tre mått tas upp i avsnitt 3.2 och används för att utvärdera hur bra en regressionsmodell presterar. Gemensamt för alla tre är att ett lägre värde betyder en bättre modell.

**RMSE** (Root Mean Squared Error, avsnitt 3.2.1, ekvation 3.1):

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Det här är nog det mått som används mest i praktiken, eftersom det ger felet i samma enhet som $y$ (t.ex. kronor), vilket gör det lätt att relatera till.

MSE (Mean Squared Error, avsnitt 3.2.2, ekvation 3.2):

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

MSE är i princip RMSE innan man drar kvadratroten. Genom att kvadrera felen slipper man att positiva och negativa fel tar ut varandra, men samtidigt blir enheten "kvadrerad" (t.ex. kr²), vilket gör talet svårare att förstå rakt av.

MAE (Mean Absolute Error, avsnitt 3.2.3, ekvation 3.3):

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

Här använder man absolutbeloppet istället för att kvadrera, av samma anledning som ovan – så att positiva och negativa fel inte tar ut varandra. Skillnaden mot RMSE är att MAE straffar alla fel linjärt oavsett storlek, medan RMSE slår hårdare mot stora avvikelser eftersom felen kvadreras.

Boken visar skillnaden med ett konkret exempel: en modell gör felet 15, en annan gör felet 75 (fem gånger så stort). Då ökar MAE också med exakt faktor 5 (från 7.5 till 37.5), medan RMSE bara ökar med en faktor på ungefär 5.89 (från 7.9 till 46.5) – ett bra exempel på att RMSE är känsligare för stora avvikelser än MAE.

```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

y_true = [3, -0.5, 2, 7]
y_pred = [2.5, 0.0, 2, 8]

mean_squared_error(y_true, y_pred)   # 0.375
mean_absolute_error(y_true, y_pred)  # 0.5
```

## Fråga 3 – Spelar det någon roll om RMSE eller MSE används för rangordning?

Nej, för själva rangordningen spelar det ingen roll om man använder RMSE eller MSE. Boken nämner i avsnitt 3.2.2 att det generellt är mer effektivt att använda MSE i sådana fall, av en ganska praktisk anledning: MSE är beräkningsmässigt billigare eftersom man slipper dra kvadratroten, och rangordningen mellan modellerna blir ändå exakt densamma oavsett vilket av måtten man väljer.

Anledningen till att ordningen inte påverkas är att RMSE bara är kvadratroten av MSE ($RMSE = \sqrt{MSE}$), och kvadratroten är en strikt växande funktion. Har modell A lägre MSE än modell B så kommer den också alltid ha lägre RMSE – inget ändrar på den ordningen.

Det som faktiskt skiljer de två måtten åt är tolkningsbarheten. RMSE ligger i samma enhet som $y$, så det är lättare att förstå hur stort felet faktiskt är i praktiken (som boken skriver i avsnitt 3.2.1: "Generellt sett använder vi RMSE om vi vill kunna tolka hur stort fel en modell gör"). MSE å sin sida är bara snabbare och billigare att räkna ut när man enbart ska jämföra modeller mot varandra.

## Fråga 4 – Förklara mycket översiktligt vad gradient descent är

**Gradient descent** (avsnitt 3.3.2) är en optimeringsalgoritm som används för att hitta minimum, eller ibland maximum, för en funktion. I maskininlärning handlar det oftast om att hitta minimum för en loss function/cost function, till exempel MSE – alltså de parametervärden ($\hat\theta_0, \hat\theta_1, \ldots$) som gör att felet mellan sanna och predikterade värden blir så litet som möjligt.

Tanken bakom är egentligen ganska enkel: för att ta sig mot ett minimum rör man sig i den riktning där lutningen är negativ, och fortsätter tills lutningen når 0, vilket är precis det som kännetecknar en minimipunkt. Algoritmen bygger på några grundbeståndsdelar:

- Slumpmässig initialisering – man måste börja någonstans, så startpunkten väljs slumpmässigt.
- Learning step/learning rate – hur stora steg som tas i varje iteration.
- Antal iterationer – hur många steg algoritmen max tillåts ta.

För linjär regressions MSE går det att visa matematiskt att kostnadsfunktionen är konvex, det vill säga att den bara har ett globalt minimum. Då vet man att minimumet man hittar faktiskt är det bästa möjliga. Så är det inte alltid för andra modeller – där kan algoritmen fastna i ett lokalt minimum, hamna på en platå (t.ex. om stegen är för små eller iterationerna för få), eller hoppa fram och tillbaka runt optimum om stegen är för stora.

Boken nämner också att det är bra att skala eller standardisera variablerna (till exempel med `StandardScaler`) innan man kör gradient descent, eftersom algoritmen då lättare hittar rätt väg mot optimum (se Figur 3.8). I praktiken sköter bibliotek som scikit-learn de här beräkningarna åt en, men det är ändå bra att ha en känsla för hur algoritmen fungerar under huven eftersom den dyker upp överallt inom ML.

## Fråga 5 – Vad är the bias variance trade-off?

**The bias variance trade-off** (avsnitt 3.3.3) är ett av de mer centrala teoretiska resultaten inom statistik och maskininlärning. Grundidén är att felet en modell gör på ny, osedd data går att dela upp i tre delar:

1. Bias – fel som kommer från felaktiga antaganden i modellen. Antar man till exempel att sambandet är linjärt fast det egentligen är kvadratiskt får man bias, vilket kan leda till underanpassning (underfitting).
2. Variance – fel som beror på att modellen är känslig för små förändringar i träningsdatan. Flexibla modeller, som högre-gradig polynomregression, kan ändras kraftigt av små skillnader i datan, vilket ger hög varians och risk för överanpassning (overfitting).
3. Irreducerbart fel – ren slumpmässighet i datan som man inte kan göra något åt oavsett modell.

Ökar man en modells komplexitet, till exempel går från enkel linjär regression till polynomregression, minskar biasen generellt medan variansen ökar – och tvärtom om man förenklar modellen. Det är därför man kallar det en trade-off: en ökning i varians kan i slutiänden kosta mer än vad man vinner på den lägre biasen.

Så varför är inte en mer komplex modell alltid bättre? Just av den anledningen. En flexibel modell fångar träningsdatan bättre, alltså lägre bias, men blir samtidigt instabil och överkänslig för små ändringar i datan, det vill säga högre varians, vilket ofta gör att den generaliserar sämre till ny data. Boken visar det här i Figur 3.9, där en enkel linjär modell (låg varians, högre bias) jämförs med en polynommodell (låg bias, hög varians) på två olika träningsdataset – den flexibla modellen ser helt olika ut mellan de två dataseten medan den enkla modellen knappt rör sig. För att faktiskt avgöra vilken modell som är bäst måste man utvärdera dem på valideringsdata.

Det här är också grunden till varför man ibland vill regularisera modeller (ridge, lasso, elastic net och liknande) – man accepterar lite högre bias i hopp om en större minskning i varians.

## Fråga 6 – Översiktlig förklaring av regressionsmodellerna

**a) Linjär regression** (avsnitt 3.3.1)

Den enklaste och mest grundläggande regressionsmodellen. Den predikterar $\hat y = \hat\theta_0 + \hat\theta_1 x$, alltså räta linjens ekvation, där $\hat\theta_0$ är interceptet och $\hat\theta_1$ lutningen. Parametrarna skattas genom att minimera MSE, antingen via gradient descent eller en analytisk lösning. Har man flera oberoende variabler kallas det multipel linjär regression.

**b) Ridge regression** (avsnitt 3.3.4, L2-regularisering)

En regulariserad variant av linjär regression som lägger till ett strafftext $\alpha \frac{1}{2}\sum \theta_i^2$ till kostnadsfunktionen. Det gör att vikterna krymps mot noll (men sätts sällan exakt till noll), vilket sänker modellens varians på bekostnad av lite högre bias. Hur hårt man regulariserar styrs av hyperparametern `alpha`, som väljs med grid search.

**c) Lasso regression** (avsnitt 3.3.5, L1-regularisering, "Least Absolute Shrinkage and Selection Operator")

Fungerar ungefär som ridge men straffar med $\alpha\sum|\theta_i|$ istället för kvadrerade vikter. Grejen med den här formen är att den faktiskt kan sätta vikterna för mindre viktiga variabler till exakt 0, vilket gör att man får variabelselektion (feature selection) på köpet – praktiskt vid högdimensionella dataset.

**d) Elastic net** (avsnitt 3.3.6)

En kombination av lasso (L1) och ridge (L2), styrd av en mix ratio $r$ (hyperparametern `l1_ratio`): $r=0$ ger ridge, $r=1$ ger lasso, och allt däremellan blandar de två. På så sätt kan modellen både välja bort variabler som lasso och samtidigt behålla små, ej-nollställda vikter som ridge.

**e) Support vector machines (SVM)** (avsnitt 3.3.7)

För regression försöker SVM lägga en "väg" med bredden $\epsilon$ runt de predikterade värdena, så att så många observationer som möjligt hamnar innanför. Observationer inom vägen (fel mindre än $\epsilon$) ignoreras helt vid träningen, modellen sägs vara "$\epsilon$-insensitive". Ett mindre $\epsilon$ ger en mer flexibel modell, ett större ett mindre flexibelt. Man bör standardisera datan innan, och modellen passar bäst för små till medelstora dataset.

**f) Beslutsträd** (avsnitt 3.3.8)

Predikterar genom att gå från rotnoden, via ett antal inre noder som var och en ställer en fråga i stil med "är $x_i \le$ tröskelvärde?", till en lövnod vars `value` blir den slutgiltiga prediktionen. Trädet väljer självt vilken variabel och vilket tröskelvärde varje fråga ska använda genom att minimera MSE i varje steg. Bör regulariseras, till exempel med `max_depth`, annars överanpassas det lätt.

**g) Ensemble learning** (avsnitt 3.3.9)

Går ut på att kombinera flera modeller i hopp om att de tillsammans presterar bättre än var och en för sig. Två exempel är voting regression, där man helt enkelt tar medelvärdet av flera olika modellers prediktioner, och bagging/pasting, där samma typ av modell tränas på flera olika delmängder av datan (se Fråga 8).

**h) Random forest** (avsnitt 3.3.10)

Ett ensemble av beslutsträd, oftast byggt genom bagging (även om pasting också förekommer). Istället för ett enda träd som lätt överanpassas tränar man många träd på olika slumpmässiga urval av datan och slår ihop deras prediktioner (medelvärde), vilket generellt ger bättre generalisering än ett enskilt träd. Precis som vanliga beslutsträd bör random forest regulariseras.

## Fråga 7 – Vad menas med white box- och black box-modeller?

I informationsrutan om modelltolkning i avsnitt 3.3.8 delar boken in modeller i två läger. White box-modeller är sådana som är intuitiva och där man lätt kan se varför modellen gör en viss prediktion. Beslutsträd är det klassiska exemplet – man kan bokstavligen stega igenom trädet fråga för fråga och se exakt vilken lövnod, och därmed vilken prediktion, en observation hamnar i.

Black box-modeller är däremot sådana där man visserligen kan följa alla beräkningar rent matematiskt och se vilken prediktion som kommer ut, men där det är svårt att i ord förklara varför modellen landade just där. Random forest och neurala nätverk är exempel boken tar upp – till skillnad från ett enskilt beslutsträd går det inte lika enkelt att manuellt följa "resonemanget" i en random forest.

Poängen är egentligen bara **tolkningsbarhet**: white box-modeller är transparenta och lätta att förklara, medan black box-modeller kan prestera bättre men är svårare att se igenom.

## Fråga 8 – Vad är skillnaden mellan bagging och pasting?

Både bagging och pasting (avsnitt 3.3.9) handlar om att skapa flera nya dataset genom slumpmässigt urval ur ett ursprungligt dataset, träna en modell per skapat dataset och sedan slå ihop prediktionerna, till exempel genom medelvärde (se Figur 3.17). Det som skiljer dem åt är hur själva urvalet görs:

- **Bagging** – urvalet görs med återläggning (*bootstrap aggregating*), så samma observation kan förekomma flera gånger i ett och samma skapade dataset. I scikit-learns `BaggingRegressor` motsvarar det `bootstrap=True`, som också är standardvärdet. En sidoeffekt av det här är att ungefär 37 % av observationerna aldrig blir dragna till ett givet dataset – de kallas *out-of-bag observations* och kan användas för att utvärdera modellen med `oob_score=True`.
- **Pasting** – urvalet görs utan återläggning, så samma observation kan inte förekomma mer än en gång i samma dataset. Motsvaras av `bootstrap=False`.

Båda kan dessutom kombineras med att bara använda en slumpmässig delmängd av de oberoende variablerna för varje dataset. Använder man hela datasetet men bara en delmängd av variablerna kallas det *random subspaces*, och kombinerar man delmängder av både observationer och variabler kallas det *random patches*. Random forest är för övrigt det vanligaste exemplet på en modell som byggs genom just bagging av beslutsträd.

## Fråga 9 (Resonångsfråga) – Tolka Figur 3.1 på sidan 113

Figur 3.1 är en schematisk bild av den enkla linjära regressionsmodellen. Koordinatsystemet har inkomst på y-axeln och ålder på x-axeln, kopplat till exempeldatan i Tabell 3.2 lite tidigare i avsnittet.

När jag tittar på figuren ser jag tre saker som samspelar:

- De svarta punkterna är den faktiska datan, $y_i$ – varje punkt är en person med en viss ålder och inkomst.
- Den blåa linjen är modellens predikterade värden, $\hat y = \hat\theta_0 + \hat\theta_1 x$. $\hat\theta_0$ (interceptet) är var linjen skär y-axeln, alltså vid ålder = 0, och $\hat\theta_1$ (lutningen) visar hur mycket den predikterade inkomsten ökar för varje extra års ålder.
- De röda vertikala linjerna är residualerna, skillnaden mellan sant och predikterat värde för varje observation: $e_i = y_i - \hat y_i$. Man ser tydligt att vissa punkter ligger långt från linjen (stor residual) medan andra ligger nästan an mot den (litet fel).

Det jag tar med mig från figuren är att den linjära regressionsmodellen försöker hitta den räta linje som bäst beskriver det generella sambandet mellan $x$ och $y$ – i det här fallet att äldre personer tenderar att tjäna mer – genom att välja $\hat\theta_0$ och $\hat\theta_1$ så att de röda residualerna sammantaget blir så små som möjligt. Det är exakt det MSE-minimeringen/gradient descent gör (se Fråga 4). Linjen fångar trenden men går inte genom varje punkt – modellen är en förenkling, och därför finns residualerna kvar.

## Fråga 10 (Resonångsfråga) – Tolka Figur 3.13 på sidan 140 och koppling till Figur 3.14 på sidan 141

Figur 3.13 visar ett tränat beslutsträd (djup 2) med två oberoende variabler, $X1$ och $X2$. Varje nod ställer en fråga i stil med "är variabeln $\le$ tröskelvärde?", och visar dessutom `squared_error` (MSE i noden), `samples` (antal observationer där) och `value` (prediktionen noden representerar).

- Rotnoden frågar "$X2 \le 0.438$?". Är svaret sant går man vänster, är det falskt går man höger.
- Vänster gren frågar vidare "$X2 \le -0.755$?", höger gren frågar "$X1 \le -0.654$?".
- De fyra lövnoderna längst ner ger de slutgiltiga prediktionerna: $-62.954$, $-8.168$, $1.837$ och $62.802$.

Boken kör igenom ett exempel: för en observation med $x_1=-1$ och $x_2=0.55$ blir svaret på "$X2 \le 0.438$?" falskt (0.55 är ju större än 0.438), så man går höger. Nästa fråga, "$X1 \le -0.654$?", blir sann (eftersom $-1 \le -0.654$), så man går vänster och hamnar i lövnoden med `value = 1.837` – det blir den slutgiltiga prediktionen.

Det som händer när man går över till Figur 3.14 är att samma träd ritas upp på ett annat sätt: som decision boundaries direkt i $(X1, X2)$-planet istället för som ett träddiagram. De tre tröskelvärdena från Figur 3.13 ($X2 = 0.438$, $X2 = -0.755$ och $X1 = -0.654$) syns nu som räta linjer som delar in planet i fyra rektangulära regioner, en per lövnod. Varje region är färglagd efter det predikterade värdet för sin lövnod (blått för lågt/negativt, rött för högt/positivt, enligt färgskalan "Predicted y"), medan prickarnas färg visar de sanna värdena, så man kan se med ögonmåttet hur väl modellens regioner stämmer med verkligheten.

Så som jag ser det är slutsatsen att ett beslutsträd delar upp variabelrummet i ett antal rektangulära regioner, en per lövnod, och att alla observationer som hamnar i samma region får samma prediktion. Träddiagrammet och regionkartan är bara två olika sätt att visa exakt samma modell.

## Fråga 11 (Resonångsfråga) – Determinationskoefficienten (R²)

Boken går igenom RMSE, MSE och MAE i avsnitt 3.2 men tar faktiskt inte upp R² i det här kapitlet. Det som följer bygger därför på allmän statistik/ML-kunskap utanför boken, men jag har försökt hålla mig till samma begrepp som boken använder (residualer, MSE och så vidare).

**Determinationskoefficienten** $R^2$ mäter hur stor andel av variationen i $y$ som modellen faktiskt lyckas förklara, till skillnad från RMSE/MSE/MAE som mäter felets storlek rakt av. Den definieras som

$$R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat y_i)^2}{\sum_{i=1}^n (y_i - \bar y)^2} = 1 - \frac{SS_{res}}{SS_{tot}}$$

där täljaren ($SS_{res}$) är summan av modellens kvadrerade fel, samma sak som ingår i MSE fast utan att delas med $n$, och nämnaren ($SS_{tot}$) är summan av kvadrerade avvikelser från medelvärdet $\bar y$ – alltså felet en helt naiv modell (som bara gissar medelvärdet varje gång) skulle göra.

Tolkningen blir ungefär så här:

- $R^2 = 1$ betyder att modellen förklarar all variation i $y$ – perfekt anpassning, inga residualer kvar.
- $R^2 = 0$ betyder att modellen inte är bättre än att bara gissa medelvärdet för varje observation.
- $R^2 < 0$ är faktiskt möjligt (till skillnad från RMSE/MSE/MAE som aldrig blir negativa) och betyder att modellen presterar sämre än att bara gissa medelvärdet, till exempel vid kraftig överanpassning.

Det som skiljer R² från RMSE, MSE och MAE är egentligen att det är skalfritt – ett tal, oftast mellan 0 och 1, utan enhet, som beskriver förklaringsgrad snarare än felstorlek. Det gör det bra att kommunicera "hur bra" en modell är i mer procent-liknande termer, till exempel för en icke-teknisk publik, medan man fortsätter behöva RMSE för att förstå hur stort felet faktiskt är i kronor, kilogram eller vad det nu må vara. I scikit-learn räknas det ut med `sklearn.metrics.r2_score`, och det är även standardmåttet som `.score()` returnerar för regressionsmodeller.

## Fråga 13 (Koduppgift) – EDA på hr_employee_data.xlsx

Här kör jag en exploratory data analysis (EDA) på datasetet `hr_employee_data.xlsx`, utan någon modellering, precis som uppgiften efterfrågar. Tanken är att en ledningsgrupp ska kunna ta del av analysen, så jag lägger vikten på tydliga, tolkningsbara visualiseringar och sammanfattande statistik snarare än avancerad teknisk output.

Analysen omfattar bland annat:

- Grundläggande översikt (`.info()`, `.describe()`, dimension, datatyper).
- Kontroll av saknade värden och dubbletter.
- Fördelningar för numeriska variabler (t.ex. ålder, lön, anställningstid) via histogram och boxplottar, för att fånga eventuella outliers.
- Fördelningar för kategoriska variabler (t.ex. avdelning, kön, anställningsstatus) via stapeldiagram.
- Korrelationsanalys mellan numeriska variabler via en korrelationsmatris/heatmap.
- Grupperade analyser (t.ex. medellön per avdelning, personalomsättning per avdelning) för att hitta mönster som är relevanta för ledningen.

OBS: Filen `hr_employee_data.xlsx` finns inte lokalt i den här miljön – den behöver hämtas från bokens GitHub-repo (https://github.com/AntonioPrgomet/ai_tillaempad_ml) innan cellen nedan kan köras. Koden nedan är därför inte exekverad.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filen behöver hämtas från bokens GitHub-repo och läggas i samma mapp som notebooken
df = pd.read_excel("hr_employee_data.xlsx")

# --- 1. Grundläggande översikt ---
print(df.shape)
df.head()
df.info()
df.describe(include="all")

# --- 2. Saknade värden och dubbletter ---
print("Antal saknade värden per kolumn:")
print(df.isnull().sum())
print("Antal dubbletter:", df.duplicated().sum())

# --- 3. Fördelningar för numeriska variabler ---
numeric_cols = df.select_dtypes(include="number").columns

fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(10, 4 * len(numeric_cols)))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i, 0])
    axes[i, 0].set_title(f"Fördelning: {col}")

    sns.boxplot(x=df[col], ax=axes[i, 1])
    axes[i, 1].set_title(f"Boxplot (outliers): {col}")
plt.tight_layout()
plt.show()

# --- 4. Fördelningar för kategoriska variabler ---
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    plt.figure(figsize=(8, 4))
    df[col].value_counts().plot(kind="bar")
    plt.title(f"Antal per kategori: {col}")
    plt.ylabel("Antal anställda")
    plt.tight_layout()
    plt.show()

# --- 5. Korrelationsanalys ---
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korrelationsmatris mellan numeriska variabler")
plt.show()

# --- 6. Grupperade analyser relevanta för ledningsgruppen ---
# Exempel: medellön per avdelning (kolumnnamn behöver anpassas efter faktiska kolumner i datasetet)
if "Department" in df.columns and "MonthlyIncome" in df.columns:
    dept_salary = df.groupby("Department")["MonthlyIncome"].mean().sort_values(ascending=False)
    dept_salary.plot(kind="bar", figsize=(8, 4), title="Medellön per avdelning")
    plt.ylabel("Medellön")
    plt.tight_layout()
    plt.show()

# Exempel: personalomsättning (attrition) per avdelning
if "Department" in df.columns and "Attrition" in df.columns:
    attrition_rate = df.groupby("Department")["Attrition"].apply(lambda x: (x == "Yes").mean())
    attrition_rate.plot(kind="bar", figsize=(8, 4), title="Andel som slutat per avdelning")
    plt.ylabel("Andel (%)")
    plt.tight_layout()
    plt.show()

# Sammanfattande nyckeltal att presentera för ledningsgruppen
print("Antal anställda totalt:", len(df))
if "MonthlyIncome" in df.columns:
    print("Medellön totalt:", df["MonthlyIncome"].mean())
if "Attrition" in df.columns:
    print("Total personalomsättning:", (df["Attrition"] == "Yes").mean())

## Fråga 16 (Koduppgift) – Komplett ML-flöde för diamantpriser (diamonds.csv)

Här kör jag ett komplett ML-flöde för att modellera diamantpriser med hjälp av datasetet `diamonds.csv` (https://www.kaggle.com/datasets/shivam2503/diamonds). Flödet följer samma struktur som modellexemplen i kapitlet:

1. Inläsning och EDA – grundläggande översikt, saknade värden, fördelningar, korrelationer mot målvariabeln `price`.
2. Hantering av kategoriska variabler – `cut`, `color` och `clarity` är ordinala, de har en inneboende rangordning, så de kodas med ordinal encoding (i rätt ordning) istället för one-hot-encoding, precis som boken beskriver för kategorisk data av den här typen.
3. Train/validation/test-uppdelning (60-20-20, se Kapitel 1).
4. Träning av ett par olika regressionsmodeller (linjär regression samt en mer flexibel modell, random forest) via `GridSearchCV`.
5. Utvärdering med RMSE på valideringsdata för att välja bästa modell, och till sist på testdata för en slutgiltig, rättvis skattning.

OBS: Filen `diamonds.csv` finns inte lokalt i den här miljön – den behöver laddas ner från Kaggle (https://www.kaggle.com/datasets/shivam2503/diamonds) innan cellen nedan kan köras. Koden nedan är därför inte exekverad.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# --- 1. Inläsning och EDA ---
df = pd.read_csv("diamonds.csv")
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(df.shape)
df.info()
df.describe()
print("Saknade värden:\n", df.isnull().sum())

sns.histplot(df["price"], kde=True)
plt.title("Fördelning av diamantpriser")
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Korrelation mellan numeriska variabler och pris")
plt.show()

# --- 2. Hantering av kategoriska (ordinala) variabler ---
# cut, color och clarity har en tydlig inbördes rangordning (låg -> hög kvalitet)
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]           # J (sämst) -> D (bäst)
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]  # I1 (sämst) -> IF (bäst)

ordinal_cols = ["cut", "color", "clarity"]
numeric_cols = ["carat", "depth", "table", "x", "y", "z"]

preprocessor = ColumnTransformer([
    ("ordinal", OrdinalEncoder(categories=[cut_order, color_order, clarity_order]), ordinal_cols),
    ("scaler", StandardScaler(), numeric_cols),
])

X = df.drop(columns=["price"])
y = df["price"]

# --- 3. Train / validation / test-uppdelning (60-20-20) ---
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

# --- 4. Träning av ett par olika regressionsmodeller ---
lin_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
lin_pipeline.fit(X_train, y_train)

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])
rf_hyperparams = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
}
rf_grid_search = GridSearchCV(estimator=rf_pipeline, param_grid=rf_hyperparams,
                               scoring="neg_mean_squared_error", cv=5)
rf_grid_search.fit(X_train, y_train)

# --- 5. Utvärdering med RMSE på valideringsdata ---
lin_val_pred = lin_pipeline.predict(X_val)
rf_val_pred = rf_grid_search.predict(X_val)

rmse_lin_val = root_mean_squared_error(y_val, lin_val_pred)
rmse_rf_val = root_mean_squared_error(y_val, rf_val_pred)

print("RMSE (linjär regression) på valideringsdata:", rmse_lin_val)
print("RMSE (random forest) på valideringsdata:", rmse_rf_val)

# Välj den modell med lägst RMSE på valideringsdata och utvärdera slutgiltigt på testdata
best_pipeline = rf_grid_search if rmse_rf_val < rmse_lin_val else lin_pipeline
test_pred = best_pipeline.predict(X_test)
rmse_test = root_mean_squared_error(y_test, test_pred)

print("Bästa modell utvärderad på testdata, RMSE:", rmse_test)